In [1]:
%pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


# 1. Business understanding

As data company provides data of commonly together sold products. Every sale is a row and columns are product categories. ID is individual sale.

Company wants to create a recommended or commonly sold together suggestion to increase sales renevue. We need to find rules for commonly together sold product categories and provide a recommender algorithm with its values to improve sales. We can use `APriori` algorithm to find frequent items and use `assosiation rules` to find rules to give as good as possible recommender system.

# 2. Data understanding

First lets look at the data given by the company. We have a comma separated values of 20 different product categories and 100000 different sales.

In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('drone_prod_groups.csv')

df

,ID,Prod1,Prod2,Prod3,Prod4,Prod5,Prod6,Prod7,Prod8,Prod9,...,Prod11,Prod12,Prod13,Prod14,Prod15,Prod16,Prod17,Prod18,Prod19,Prod20
0,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1
1,2,0,1,0,0,0,0,0,0,1,...,0,0,0,0,1,1,1,1,1,1
2,3,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,1
3,4,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,1
4,5,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99996,99997,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
99997,99998,0,1,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
99998,99999,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1


In [4]:
print('Dataframes sum of NA values:')
print(df.isna().sum())
print('All NA values: ', df.isna().sum().sum())
print('\nDataframe dataypes:')
print(df.dtypes)

Dataframes sum of NA values:
ID         0
Prod1      0
 Prod2     0
 Prod3     0
 Prod4     0
 Prod5     0
 Prod6     0
 Prod7     0
 Prod8     0
 Prod9     0
 Prod10    0
 Prod11    0
 Prod12    0
 Prod13    0
 Prod14    0
 Prod15    0
 Prod16    0
 Prod17    0
 Prod18    0
 Prod19    0
 Prod20    0
dtype: int64
All NA values:  0

Dataframe dataypes:
ID         int64
Prod1      int64
 Prod2     int64
 Prod3     int64
 Prod4     int64
 Prod5     int64
 Prod6     int64
 Prod7     int64
 Prod8     int64
 Prod9     int64
 Prod10    int64
 Prod11    int64
 Prod12    int64
 Prod13    int64
 Prod14    int64
 Prod15    int64
 Prod16    int64
 Prod17    int64
 Prod18    int64
 Prod19    int64
 Prod20    int64
dtype: object


## Data breakdown

Rows are commonly together sold product categories. Value `1` indicates as sold in one purchase and `0` indicates not sold.

No empty values in data, so there is no need to fill. All values are `1` or `0` so they can be converted as boolean values `true` or `false` and no need for scaling.

# 3. Data preparation

First we drop `ID` column from dataset and convert `1` as `true` and `0` as `false`. Scaling is not necessary and no need to fill empty values as non sold items are indicated with `0`. Whole data is in one dataset so no need to combine.

In [5]:
df = df.drop(columns='ID')

df = df.astype(bool)

# 4. Modeling

We use `APriori` algorithm to get a `frequent itemset`. Lets choose `min_support` value as a low to get enough itemsets.

In [6]:
frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.10998,frozenset({Prod1})
1,0.13098,frozenset({ Prod2})
2,0.03271,frozenset({ Prod3})
3,0.03585,frozenset({ Prod4})
4,0.10459,frozenset({ Prod5})
...,...,...
165,0.02030,"frozenset({ Prod20, Prod19, Prod15})"
166,0.02203,"frozenset({ Prod20, Prod19, Prod16})"
167,0.02052,"frozenset({ Prod20, Prod19, Prod18})"
168,0.01101,"frozenset({ Prod20, Prod19, Prod12, Prod5})"


# 5. Evaluation

We need to find sufficient `support`, `lift` and `confidence` values to provide enough itemsets, but accurate enough to get good enough sales recommendations.

## Finding support value based on confidence and lift metrics

Lets try multiple support values and see the `rules` and `itemsets` number found.

In [7]:
support_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

results = []

for min_supp in support_values:
    
    frequent_itemsets = apriori(
        df,
        min_support=min_supp,
        use_colnames=True
    )

    rules_test = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=0.5
    )

    results.append({
        'min_support': min_supp,
        'frequent_itemsets': len(frequent_itemsets),
        'rules': len(rules_test)
    })

support_test_conf = pd.DataFrame(results)

In [8]:
support_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

results = []

for min_supp in support_values:
    
    frequent_itemsets = apriori(
        df,
        min_support=min_supp,
        use_colnames=True
    )

    rules_test = association_rules(
        frequent_itemsets,
        metric='lift',
        min_threshold=0.5
    )

    results.append({
        'min_support': min_supp,
        'frequent_itemsets': len(frequent_itemsets),
        'rules': len(rules_test)
    })

support_test_lift = pd.DataFrame(results)

In [9]:
display(support_test_conf)

display(support_test_lift)

,min_support,frequent_itemsets,rules
0,0.001,1564,888
1,0.002,903,551
2,0.005,339,133
3,0.010,170,77
4,0.020,97,31
5,0.050,19,5


,min_support,frequent_itemsets,rules
0,0.001,1564,15216
1,0.002,903,6598
2,0.005,339,1506
3,0.010,170,472
4,0.020,97,208
5,0.050,19,6


## Finding confidence value

Lets try multiple confidence values and see the number of `rules` and when they start to decrease and by how much.

In [10]:
# Frequent itemsets with chose min support value of 0.01
frequent_itemsets = apriori(
    df,
    min_support=0.01,
    use_colnames=True
)

In [11]:
confidence_values = [0.3, 0.4, 0.425, 0.45, 0.5, 0.525, 0.55, 0.575, 0.6, 0.7, 0.8]

results = []

for min_conf in confidence_values:

    rules_test = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=min_conf
    )

    results.append({
        'min_confidence': min_conf,
        'rules': len(rules_test)
    })

confidence_test = pd.DataFrame(results)

display(confidence_test)

,min_confidence,rules
0,0.300,91
1,0.400,89
2,0.425,81
3,0.450,77
4,0.500,77
5,0.525,76
6,0.550,76
7,0.575,63
8,0.600,60
9,0.700,32


## Finding lift value

Lets try multiple lift values and see the number `rules` and when it starts to decrease to find the highest lift value.

In [12]:
lift_values = [1, 1.2, 1.5, 2, 3, 4, 4.25, 4.5, 4.75]

results = []

for min_lift in lift_values:

    rules_test = association_rules(
        frequent_itemsets,
        metric='lift',
        min_threshold=min_lift
    )

    results.append({
        'min_lift': min_lift,
        'rules': len(rules_test)
    })

lift_test = pd.DataFrame(results)

display(lift_test)

,min_lift,rules
0,1.00,472
1,1.20,426
2,1.50,170
3,2.00,170
4,3.00,170
5,4.00,160
6,4.25,128
7,4.50,82
8,4.75,38


## Conclusion

### Support

Based on test results `support value` for model good option is `0.01`. Based on confidence does not agressively exclude `rules` (77) and includes sufficient amount of `frequent itemsets` (170).

### Confidence

Rules start to drop on `confidence` value of 0.550 from 5 `rules` to 4 when raising value to 0.575 and 0.550 value is confident enough to recommend enought itemsets.

### Lift

With `lift` value, the drop is higher when it raises above 4.00. It drops from 160 rules to 128 rules from `lift` value of 4.00 to 4.25. This is higher than the 170 to 160 drop with higher `lift` value drop from 3 to 4.

In [13]:
rules_conf = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.55)
rules_supp = association_rules(frequent_itemsets, metric='support', min_threshold=0.01)
rules_lift = association_rules(frequent_itemsets, metric='lift', min_threshold=4)

rules_conf = rules_conf.sort_values(by='confidence', ascending=False)
rules_supp = rules_supp.sort_values(by='support', ascending=False)
## Lift(A ->B) = supp(A or B) / ( supp(A) * supp(B) ) = conf(A -> B) / supp(B) || 1 -> riippumattomia ja sit posi ja neg korrelaatiot
rules_lift = rules_lift.sort_values(by='lift', ascending=False)

In [14]:
rules_conf

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
11,"frozenset({ Prod2, Prod15})",frozenset({ Prod9}),0.01947,0.19853,0.01843,0.946584,4.767967,1.0,0.014565,15.004443,0.805959,0.092349,0.933353,0.519708
49,"frozenset({ Prod20, Prod15})",frozenset({ Prod9}),0.02241,0.19853,0.02119,0.945560,4.762807,1.0,0.016741,14.722084,0.808150,0.106083,0.932075,0.526147
73,"frozenset({ Prod20, Prod19, Prod15})",frozenset({ Prod9}),0.02030,0.19853,0.01919,0.945320,4.761599,1.0,0.015160,14.657514,0.806356,0.096123,0.931776,0.520990
39,"frozenset({ Prod15, Prod12})",frozenset({ Prod9}),0.02308,0.19853,0.02173,0.941508,4.742396,1.0,0.017148,13.702169,0.807780,0.108715,0.927019,0.525481
27,"frozenset({ Prod15, Prod7})",frozenset({ Prod9}),0.02014,0.19853,0.01895,0.940914,4.739403,1.0,0.014952,13.564375,0.805220,0.094883,0.926277,0.518183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,"frozenset({ Prod9, Prod14})",frozenset({ Prod15}),0.03484,0.11880,0.01969,0.565155,4.757197,1.0,0.015551,2.026469,0.818302,0.146995,0.506531,0.365448
38,"frozenset({ Prod9, Prod12})",frozenset({ Prod15}),0.03845,0.11880,0.02173,0.565150,4.757151,1.0,0.017162,2.026444,0.821372,0.160345,0.506525,0.374031
30,"frozenset({ Prod9, Prod8})",frozenset({ Prod15}),0.03893,0.11880,0.02195,0.563833,4.746065,1.0,0.017325,2.020325,0.821271,0.161659,0.505030,0.374298
1,frozenset({ Prod9}),frozenset({ Prod15}),0.19853,0.11880,0.11145,0.561376,4.725388,1.0,0.087865,2.009011,0.983664,0.541335,0.502243,0.749754


In [15]:
rules_supp

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
221,frozenset({ Prod19}),frozenset({ Prod20}),0.20626,0.14798,0.13476,0.653350,4.415125,1.0,0.104238,2.457869,0.974508,0.613997,0.593144,0.782007
220,frozenset({ Prod20}),frozenset({ Prod19}),0.14798,0.20626,0.13476,0.910664,4.415125,1.0,0.104238,8.884845,0.907849,0.613997,0.887449,0.782007
131,frozenset({ Prod15}),frozenset({ Prod9}),0.11880,0.19853,0.11145,0.938131,4.725388,1.0,0.087865,12.954372,0.894663,0.541335,0.922806,0.749754
130,frozenset({ Prod9}),frozenset({ Prod15}),0.19853,0.11880,0.11145,0.561376,4.725388,1.0,0.087865,2.009011,0.983664,0.541335,0.502243,0.749754
65,frozenset({ Prod12}),frozenset({ Prod5}),0.15971,0.10459,0.06683,0.418446,4.000822,1.0,0.050126,1.539685,0.892610,0.338431,0.350516,0.528709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,"frozenset({ Prod12, Prod18})",frozenset({ Prod5}),0.02361,0.10459,0.01033,0.437526,4.183253,1.0,0.007861,1.591915,0.779352,0.087639,0.371826,0.268147
289,"frozenset({ Prod5, Prod18})",frozenset({ Prod12}),0.01550,0.15971,0.01033,0.666452,4.172886,1.0,0.007854,2.519245,0.772329,0.062652,0.603056,0.365566
288,"frozenset({ Prod5, Prod12})",frozenset({ Prod18}),0.06683,0.12166,0.01033,0.154571,1.270519,1.0,0.002199,1.038929,0.228168,0.057982,0.037470,0.119740
191,frozenset({ Prod14}),frozenset({ Prod17}),0.14557,0.05618,0.01005,0.069039,1.228888,1.0,0.001872,1.013813,0.217989,0.052426,0.013624,0.123964


In [16]:
rules_lift

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
155,"frozenset({ Prod19, Prod15})","frozenset({ Prod9, Prod20})",0.03041,0.03676,0.01919,0.631042,17.166551,1.0,0.018072,2.610707,0.971284,0.399958,0.616962,0.576539
152,"frozenset({ Prod9, Prod20})","frozenset({ Prod19, Prod15})",0.03676,0.03041,0.01919,0.522035,17.166551,1.0,0.018072,2.028579,0.977687,0.399958,0.507044,0.576539
154,"frozenset({ Prod20, Prod15})","frozenset({ Prod9, Prod19})",0.02241,0.04996,0.01919,0.856314,17.139995,1.0,0.018070,6.611924,0.963243,0.360850,0.848758,0.620211
153,"frozenset({ Prod9, Prod19})","frozenset({ Prod20, Prod15})",0.04996,0.02241,0.01919,0.384107,17.139995,1.0,0.018070,1.587273,0.991176,0.360850,0.369989,0.620211
142,"frozenset({ Prod19, Prod12})","frozenset({ Prod20, Prod5})",0.03881,0.01888,0.01101,0.283690,15.025941,1.0,0.010277,1.369686,0.971138,0.235861,0.269906,0.433423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,"frozenset({ Prod7, Prod12})",frozenset({ Prod5}),0.02586,0.10459,0.01083,0.418794,4.004145,1.0,0.008125,1.540606,0.770176,0.090537,0.350905,0.261170
146,frozenset({ Prod12}),"frozenset({ Prod20, Prod19, Prod5})",0.15971,0.01722,0.01101,0.068937,4.003336,1.0,0.008260,1.055547,0.892797,0.066357,0.052624,0.354155
137,"frozenset({ Prod20, Prod19, Prod5})",frozenset({ Prod12}),0.01722,0.15971,0.01101,0.639373,4.003336,1.0,0.008260,2.330080,0.763353,0.066357,0.570830,0.354155
1,frozenset({ Prod12}),frozenset({ Prod5}),0.15971,0.10459,0.06683,0.418446,4.000822,1.0,0.050126,1.539685,0.892610,0.338431,0.350516,0.528709


In [17]:
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.0
)

rules = rules[
    (rules['support'] >= 0.01) &
    (rules['confidence'] >= 0.55) &
    (rules['lift'] >= 4.0)
]

rules = rules.sort_values(
    by=['confidence', 'lift', 'support'],
    ascending=False
)

display(
    rules[
        ['antecedents', 'consequents',
         'support', 'confidence', 'lift']
    ]
)

,antecedents,consequents,support,confidence,lift
242,"frozenset({ Prod2, Prod15})",frozenset({ Prod9}),0.01843,0.946584,4.767967
392,"frozenset({ Prod20, Prod15})",frozenset({ Prod9}),0.02119,0.945560,4.762807
461,"frozenset({ Prod20, Prod19, Prod15})",frozenset({ Prod9}),0.01919,0.945320,4.761599
356,"frozenset({ Prod15, Prod12})",frozenset({ Prod9}),0.02173,0.941508,4.742396
314,"frozenset({ Prod15, Prod7})",frozenset({ Prod9}),0.01895,0.940914,4.739403
...,...,...,...,...,...
367,"frozenset({ Prod9, Prod14})",frozenset({ Prod15}),0.01969,0.565155,4.757197
355,"frozenset({ Prod9, Prod12})",frozenset({ Prod15}),0.02173,0.565150,4.757151
324,"frozenset({ Prod9, Prod8})",frozenset({ Prod15}),0.02195,0.563833,4.746065
130,frozenset({ Prod9}),frozenset({ Prod15}),0.11145,0.561376,4.725388


In [18]:
max_confidence = rules['confidence'].max()

min_confidence = rules['confidence'].min()

print("Max confidence:", max_confidence)
print("Min confidence:", min_confidence)

Max confidence: 0.9465844889573701
Min confidence: 0.5583224115334207


## Conclusion

With values of `support` as 0.01, `confidence` as 0.55 and `lift` as 4 we get confident recommendations ranging from approximtely 0.56 to 0.95. Confidence being 0.55 is not the greatest but with this data it gives us sufficient item recommendations. It does not include most of the products sets but lowering support would give less reliable recommendations to the customer. Model does provide sufficient enough of data for recomender system.

# 6. Deployment

Company might want to test out different rule mining algorithms, but if they decide to use `APriori` they should use minimum support value as **`0.01`**. It gives confident enough recommendation. Company might want to filter any recommendations with under **`0.55`** `confidence`and **`4.00`** `lift` to get good enought recommendations. 

In future with new and updated data, company might want to rerun test and find new, updated values for new model. The test from evaluation can be used for this.

Model does not provide recommendations for all product groups combinations, but sufficient enough to give at least some recommendations. It might lower the confidence by too much to recommend less reliable products.